<a href="https://colab.research.google.com/github/sunmgr/ml-projects/blob/main/disaster_tweets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pandas as pd
import numpy as np
import re

In [19]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [20]:
train_df.shape

(7613, 5)

In [21]:
test_df.shape

(3263, 4)

In [22]:
train_df

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [23]:
def clean_data(text):
  if pd.isna(text):
    return ""
  text = text.lower()                                  # Lowercase
  text = re.sub(r'https?://\S+|www\.\S+', '', text)    # Remove URLs
  text = re.sub(r'<.*?>', '', text)                    # Remove HTML tags
  text = re.sub(r'[^a-z0-9\s]', '', text)              # Remove punctuation
  return text

In [24]:
train_df['clean_text'] = train_df['text'].apply(clean_data)
test_df['clean_text'] = test_df['text'].apply(clean_data)


In [25]:
train_df

,id,keyword,location,text,target,clean_text
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1,our deeds are the reason of this earthquake ma...
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1,forest fire near la ronge sask canada
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1,all residents asked to shelter in place are be...
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1,13000 people receive wildfires evacuation orde...
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1,just got sent this photo from ruby alaska as s...
...,...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1,two giant cranes holding a bridge collapse int...
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1,ariaahrary thetawniest the out of control wild...
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1,m194 0104 utc5km s of volcano hawaii
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1,police investigating after an ebike collided w...


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [27]:
vectorizer = TfidfVectorizer(max_features=10000,ngram_range=(1,2),stop_words='english')
X_train = vectorizer.fit_transform(train_df['clean_text'])
y_train = train_df['target']
X_test = vectorizer.transform(test_df['clean_text'])

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score,StratifiedKFold

In [31]:
model = LogisticRegression(C=1,max_iter=1000)
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scores = cross_val_score(model,X_train,y_train,cv=cv,scoring='f1')
print(f"Cross-Validation F1 Scores: {scores}")
print(f"Mean CV F1 Score: {scores.mean():.4f}")

Cross-Validation F1 Scores: [0.75215146 0.73666093 0.7315834  0.74585153 0.74698795]
Mean CV F1 Score: 0.7426


In [32]:
model.fit(X_train, y_train)
test_preds = model.predict(X_test)

In [33]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'target': test_preds
})

In [34]:
submission.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' generated successfully!")

Submission file 'submission.csv' generated successfully!
